# Visualizing attributions

This notebook is a tool for visualizing the attributions calculated. The attribution are calculated using the library path explain (https://github.com/suinleelab/path_explain.git), which also provide some functions for visualizing the attributions. First, import all the required libraries and the config file.

The file `explainer_utils.py` contains mappings to rename the original varaibles used in the model such that they're more readable in the plots. 

In [ ]:
from path_explain import summary_plot
import matplotlib.pyplot as plt
import numpy as np
from explainer_utils import get_vars, get_super_groups
import utils

config = utils.load_config('config.yaml')
display_names, group_keys = get_vars(config)

## Load the attributions
The attributions are calculated using the `explainer.py` script (the instructions for the usage can be found in the `README`). Now we load the npz file that contains the attributions and the samples from the data used for the calculation.  

In [ ]:
attr_file = 'attributions.npz' # path to npz file
_data = np.load(attr_file)
attributions = _data['attributions']
data = _data['data']
print(f"Loaded attributions from {attr_file} with shape: {attributions.shape}")

## Beeswarm plot

The path explain library includes a function to produce a beeswarm plot that summarizes the attributions sorted by importance, where the top variable correspondes to the highest attribution value. The x axis show the attribution value for each data point, and the color map shows the feature value for each data point. Modifying the parameter `top_k` allows to change how many variables are shown at same time in the plot. 

The colormap can be changed by using the parameter `cmap`. 

In [ ]:
top_k = 10 # number of variables to display

summary_plot(attributions=attributions, 
             plot_top_k=top_k,
             feature_values=data, 
             feature_names=display_names)

## Combining tracks

As seen in the beeswarm, the variables are separated by tracks. Combining the tracks allows to interpret the attributions given by the physical variables. Then the plot can then be interpreted as raking of the order of importance of the variables for the model. The attributions are additivie, so to find the combined attributions we just need to add the respective attributions. The variables are still separated between *trk*, *ecl* and *roe*, so it's possible to see the role played by the EM Cluster and the Rest of Event. 

Here again, it's possible to use the parameter `top_k` to control the number of variables shown. When set to `n_groups` it plots the full list of 37 variables with combined tracks.

In [ ]:
unique_groups = list(dict.fromkeys(group_keys))
group_index = {g: k for k, g in enumerate(unique_groups)}
col_to_group = np.array([group_index[g] for g in group_keys])

n_samples, _ = attributions.shape
n_groups = len(unique_groups)

combined_attr = np.zeros((n_samples, n_groups), dtype=attributions.dtype)

for c, g in enumerate(col_to_group):
    combined_attr[:, g] += attributions[:, c]

top_k = 20 # n_groups plot all the variables
mean_abs = np.mean(np.abs(combined_attr), axis=0)
order = np.argsort(mean_abs)[::-1][:top_k]

plt.figure(figsize=(8, 8))
plt.barh([unique_groups[i] for i in order][::-1], mean_abs[order][::-1])
plt.xlabel('Mean attribution  (summed over particles)')
plt.title(f'Feature importances')
plt.tight_layout()
plt.show()

## Combine variables into categories

For an even more compact plot, it's possible to combine the variables into greater categories. They are groupped by the nature of the variable, or where it comes from. The groups chosen are:
- **Charge**: electric charge. Since it has a great importance, it has its own group.
- **Kinematic**: these are the variables related to the dynamics measured by the detectors. They are separated by the *trk* and *ecl*. e.g. p (momentum), cosθ (momentum cosine of polar angle).
- **PID**: particle ID. They represent the probability of a track containing a certain particle, e.g. electron, pion, etc.
- **PID expert**: specific particle ID's. These variables are probabilities of a track containing a certain particle over another, comparing one to one. These are used to calculate the *PID* variables. 
- **Tracking**: tracking parameters measured in the detectors. e.g. nPXD (number of hits in the pixel detector).
- **Cluster**: all remaining variables measured in the EM Cluster.
- **ROE**: variables corresponding to the Rest of Event. 

The full classification can be found in `explainer_utils.py`, where the groups could be adjusted in case of requiring a different grouping. This classification is based on the official categories on the Sphinx documentation: https://software.belle2.org/development/sphinx/analysis/doc/Variables.html#variables-by-group. 

The resulting plot is the ranking of importance of each group for the model. 

In [ ]:
super_groups = get_super_groups()
label_to_super = {label: g for g, labels in super_groups.items() for label in labels}

super_names = list(super_groups)
super_idx = {g: i for i, g in enumerate(super_names)}
col_to_super = np.array([super_idx[label_to_super[g]] for g in unique_groups])

n_super = len(super_names)
combined_attr_total = np.zeros((n_samples, n_super), dtype=attributions.dtype)
for c, g in enumerate(col_to_super):
    combined_attr_total[:, g] += combined_attr[:, c]

mean_abs = np.mean(np.abs(combined_attr_total), axis=0)
order = np.argsort(mean_abs)[::-1]

plt.figure(figsize=(8, 6))
plt.barh([super_names[i] for i in order][::-1], mean_abs[order][::-1])
plt.xlabel('Mean attribution')
plt.title('Feature importances by group')
plt.tight_layout()
plt.show()

## Top 5 variables

The top variables (with tracks combined) contain 10 tracks each. The following cell produces a plot for each variable, showing the attribution fraction carried by each track for the variable. This shows the behavior of importance. Since the tracks are sorted by momentum, it is expected that the first track carries the higher importance. 

In [ ]:
parameters = config['parameters']
trk_variable_list = config['trk_variable_list']

n_trk        = parameters['num_trk']
n_trk_feats  = parameters['num_trk_features']

top5_vars = {
    'Charge':  'charge',
    'π ID':    'pionID',
    'phi CMS': 'useCMSFrame(phi)',
    'p CMS':   'useCMSFrame(p)',
    'Δz':      'dzdiff',
}

def trk_cols(var_name):
    idx = trk_variable_list.index(var_name)
    return np.array([t * n_trk_feats + idx for t in range(n_trk)])

track_ranks = np.arange(1, n_trk + 1)

even_share = 1.0 / n_trk

for label, raw in top5_vars.items():
    abs_per_track = np.abs(attributions[:, trk_cols(raw)])            
    totals        = abs_per_track.sum(axis=1, keepdims=True)   
    # Skip samples where every track contributed 0 (avoid 0/0).
    keep          = (totals.squeeze() > 0)
    frac          = abs_per_track[keep] / totals[keep]         

    plt.figure(figsize=(7, 4))
    plt.boxplot([frac[:, t] for t in range(n_trk)],
                positions=track_ranks, widths=0.6, showfliers=False)
    plt.axhline(even_share, color='red', linestyle='--', linewidth=1,
                label=f'even share')
    plt.xlabel('Track rank (by p)')
    plt.ylabel('Fraction of total attribution')
    plt.title(f'Per-sample fractional contribution per track — {label}')
    plt.xticks(track_ranks, track_ranks)
    plt.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()

## Particle ID for tracks

When using a small number of data samples, there is an unexpected behavior. It turns out that track 4 for the momentum has a greater importance than track 3. Plotting the particle ID could provide a little more information abou the origin of this artifact. This is resolved when taking more samples from the data, to get a more consistent result. However, the tool for plotting remains in here for additional information and completeness. 

In [ ]:
def trk_col(var_name, t):
    """Column index in attributions/test_x for 0-based track index t and variable name."""
    return t * n_trk_feats + trk_variable_list.index(var_name)

ranks = [3, 4] # Select the ranks to show

pid_vars  = ['pionID', 'muonID', 'electronID', 'kaonID', 'protonID']
pid_names = ['π', 'μ', 'e', 'K', 'p']

x = np.arange(len(pid_names))
w = 0.4
offsets = [-w / 2, w / 2]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for off, rank in zip(offsets, ranks):
    t = rank - 1
    cols = [trk_col(v, t) for v in pid_vars]
    pid_matrix = data[:, cols]                   

    # A track is "present" if its PID row isn't all-zero (zero = padded/absent slot).
    present = pid_matrix.any(axis=1)
    pid_present = pid_matrix[present]

    assigned = np.argmax(pid_present, axis=1)
    counts   = np.bincount(assigned, minlength=len(pid_vars))
    axes[0].bar(x + off, counts, width=w, edgecolor='black',
                label=f'track {rank} (n={present.sum()})')

    mean_pid = pid_present.mean(axis=0)
    axes[1].bar(x + off, mean_pid, width=w, edgecolor='black', label=f'track {rank}')

axes[0].set_title('Most-likely species (argmax PID)')
axes[0].set_ylabel('Track count')
axes[1].set_title('Mean PID probability')
axes[1].set_ylabel('Mean value')
for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(pid_names)
    ax.set_xlabel('Particle ID')
    ax.legend()
fig.suptitle(f'Particle identity of tracks {ranks[0]} and {ranks[1]}', y=1.02)
plt.tight_layout()
plt.show()